<a href="https://colab.research.google.com/github/rahafabumwise/IEEE-AI-Modeling-Hackathon-2.0-Stage-1-Challenge/blob/main/clean_best_model_validation_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clean Best Model + Validation Lab

This notebook keeps only the parts that matter:

- the exact 58-feature thermal/stress pipeline behind the **0.20500 Kaggle submission**
- fold-safe 5-fold OOF validation
- CatBoost, LightGBM, and fixed 80/20 blend checkpoints
- a fixed **test-like validation checkpoint** based on adversarial validation
- saved OOF predictions and a compact experiment scoreboard
- exact full-data training and submission generation

Rejected experiments such as GMM regime probabilities, regime interactions, density weighting, manual corrections, and repeated residual binning are intentionally excluded.

## Workflow

1. Run setup and load data.
2. Run the shift checkpoint once.
3. Run the exact baseline OOF section.
4. Record the baseline checkpoint.
5. Test one controlled candidate at a time.
6. Generate a submission only after a candidate passes the checkpoints.


In [1]:
!pip install -q catboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.2 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import pandas as pd
import lightgbm as lgb

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, LGBMClassifier

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_log_error, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 200)


## 1. Load data

Change only `DATA_PATH` when needed.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
DATA_PATH = "/content/drive/MyDrive/datasets/ieee-ai-modeling-hackathon2-stage-1-challenge"

train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

X_raw = train.drop(columns=["id", "edi"]).copy()
X_test_raw = test.drop(columns=["id"]).copy()

y_raw = train["edi"].copy()
y_log = np.log1p(y_raw)

print("Train:", train.shape)
print("Test:", test.shape)
print("Raw train features:", X_raw.shape)
print("Raw test features:", X_test_raw.shape)

assert train.shape == (24000, 44)
assert test.shape == (16000, 43)
assert X_raw.shape[1] == 42
assert list(X_raw.columns) == list(X_test_raw.columns)


Train: (24000, 44)
Test: (16000, 43)
Raw train features: (24000, 42)
Raw test features: (16000, 42)


## 2. Exact preprocessing and feature engineering

All learned preprocessing is fitted on the training fold only.

In [5]:
MISSING_COLS = [
    "humidity",
    "sensor_17",
    "vibration_rms",
    "coolant_flow",
    "hours_since_service",
    "sensor_05",
]


def preprocess_fold(X_train_fold, X_valid_fold):
    X_train_fold = X_train_fold.copy()
    X_valid_fold = X_valid_fold.copy()

    for col in MISSING_COLS:
        X_train_fold[f"{col}_was_missing"] = (
            X_train_fold[col].isna().astype(np.int8)
        )
        X_valid_fold[f"{col}_was_missing"] = (
            X_valid_fold[col].isna().astype(np.int8)
        )

    X_train_fold["total_missing_count"] = (
        X_train_fold[MISSING_COLS].isna().sum(axis=1)
    )
    X_valid_fold["total_missing_count"] = (
        X_valid_fold[MISSING_COLS].isna().sum(axis=1)
    )

    fold_medians = {}
    for col in MISSING_COLS:
        median_value = X_train_fold[col].median()
        fold_medians[col] = float(median_value)

        X_train_fold[col] = X_train_fold[col].fillna(median_value)
        X_valid_fold[col] = X_valid_fold[col].fillna(median_value)

    return X_train_fold, X_valid_fold, fold_medians


In [6]:
def add_engineered_features_stress(X):
    X = X.copy()

    X["load_duty_combo"] = (
        X["load_factor"] * X["duty_cycle"]
    )

    X["stress_index"] = (
        X["core_temp"]
        * X["load_factor"]
        * X["duty_cycle"]
    )

    X["thermal_excess"] = X["delta_ambient"]

    X["electrical_stress"] = (
        X["harmonic_thd"] * X["load_factor"]
    )

    X["mechanical_stress"] = (
        X["vibration_rms"] * X["load_factor"]
    )

    X["combined_operating_stress"] = (
        X["core_temp"]
        * X["load_factor"]
        * X["duty_cycle"]
        * (1.0 + X["harmonic_thd"])
    )

    return X


In [7]:
TEMP_PREDICTOR_FEATURES = [
    "asset_age",
    "load_factor",
    "duty_cycle",
    "coolant_flow",
    "humidity",
    "vibration_rms",
    "grid_freq",
    "line_voltage",
    "harmonic_thd",
    "phase_imbalance",
    "hours_since_service",
    "chamber_pressure",
]


def add_temperature_residual_features(X_train_raw, X_valid_raw):
    X_train_raw = X_train_raw.copy()
    X_valid_raw = X_valid_raw.copy()

    temperature_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ])

    temperature_model.fit(
        X_train_raw[TEMP_PREDICTOR_FEATURES],
        X_train_raw["core_temp"],
    )

    train_expected = temperature_model.predict(
        X_train_raw[TEMP_PREDICTOR_FEATURES]
    )
    valid_expected = temperature_model.predict(
        X_valid_raw[TEMP_PREDICTOR_FEATURES]
    )

    X_train_raw["expected_core_temp"] = train_expected
    X_valid_raw["expected_core_temp"] = valid_expected

    X_train_raw["core_temp_residual"] = (
        X_train_raw["core_temp"] - train_expected
    )
    X_valid_raw["core_temp_residual"] = (
        X_valid_raw["core_temp"] - valid_expected
    )

    X_train_raw["abs_core_temp_residual"] = (
        X_train_raw["core_temp_residual"].abs()
    )
    X_valid_raw["abs_core_temp_residual"] = (
        X_valid_raw["core_temp_residual"].abs()
    )

    return X_train_raw, X_valid_raw, temperature_model


In [8]:
def calculate_rmsle(actual, prediction):
    actual = np.asarray(actual)
    prediction = np.clip(np.asarray(prediction), 0, None)

    return float(
        np.sqrt(mean_squared_log_error(actual, prediction))
    )


def build_fold_features(X_train_raw, X_valid_raw):
    X_train_thermal, X_valid_thermal, temperature_model = (
        add_temperature_residual_features(
            X_train_raw,
            X_valid_raw,
        )
    )

    X_train, X_valid, medians = preprocess_fold(
        X_train_thermal,
        X_valid_thermal,
    )

    X_train = add_engineered_features_stress(X_train)
    X_valid = add_engineered_features_stress(X_valid)

    assert list(X_train.columns) == list(X_valid.columns)
    assert X_train.isna().sum().sum() == 0
    assert X_valid.isna().sum().sum() == 0
    assert X_train.shape[1] == 58
    assert X_valid.shape[1] == 58

    return X_train, X_valid, temperature_model, medians


## 3. Shift checkpoint: how different are train and test?

This section creates a cross-fitted `test_likeness_score` for every training row.

- AUC near 0.50 means train and test are difficult to distinguish.
- AUC clearly above 0.50 confirms distribution shift.
- The top 20% most test-like training rows form a fixed secondary validation checkpoint.

This checkpoint does not replace normal OOF. It complements it.

In [9]:
combined_X = pd.concat(
    [X_raw, X_test_raw],
    axis=0,
    ignore_index=True,
)

combined_origin = np.concatenate([
    np.zeros(len(X_raw), dtype=np.int8),
    np.ones(len(X_test_raw), dtype=np.int8),
])

shift_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

origin_oof_probability = np.zeros(len(combined_X))

for fold, (fit_idx, valid_idx) in enumerate(
    shift_cv.split(combined_X, combined_origin),
    start=1,
):
    shift_model = LGBMClassifier(
        objective="binary",
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=30,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED + fold,
        n_jobs=-1,
        verbosity=-1,
    )

    shift_model.fit(
        combined_X.iloc[fit_idx],
        combined_origin[fit_idx],
        eval_set=[(
            combined_X.iloc[valid_idx],
            combined_origin[valid_idx],
        )],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    origin_oof_probability[valid_idx] = shift_model.predict_proba(
        combined_X.iloc[valid_idx],
        num_iteration=shift_model.best_iteration_,
    )[:, 1]

shift_auc = roc_auc_score(
    combined_origin,
    origin_oof_probability,
)

train_test_likeness = origin_oof_probability[:len(X_raw)]

test_like_threshold = np.quantile(
    train_test_likeness,
    0.80,
)

very_test_like_mask = (
    train_test_likeness >= test_like_threshold
)

print("Adversarial validation AUC:", shift_auc)
print("Test-like train rows:", int(very_test_like_mask.sum()))
print(pd.Series(train_test_likeness).describe())


Adversarial validation AUC: 0.6398411640625
Test-like train rows: 4800
count    24000.000000
mean         0.376958
std          0.109836
min          0.139532
25%          0.292533
50%          0.364865
75%          0.449434
max          0.789407
dtype: float64


In [10]:
# Simple univariate shift report for interpretation only
from scipy.stats import ks_2samp

shift_rows = []

for col in X_raw.columns:
    train_values = X_raw[col].dropna()
    test_values = X_test_raw[col].dropna()

    ks_stat, p_value = ks_2samp(
        train_values,
        test_values,
    )

    shift_rows.append({
        "feature": col,
        "ks_stat": ks_stat,
        "p_value": p_value,
        "train_mean": train_values.mean(),
        "test_mean": test_values.mean(),
        "train_missing_pct": X_raw[col].isna().mean() * 100,
        "test_missing_pct": X_test_raw[col].isna().mean() * 100,
    })

shift_report = (
    pd.DataFrame(shift_rows)
    .sort_values("ks_stat", ascending=False)
    .reset_index(drop=True)
)

display(shift_report.head(15))


,feature,ks_stat,p_value,train_mean,test_mean,train_missing_pct,test_missing_pct
0,core_temp,0.146000,4.661104e-179,59.229526,62.990345,0.000000,0.00000
1,sensor_25,0.144854,2.980530e-176,29.953079,32.096583,0.000000,0.00000
2,delta_ambient,0.139229,8.350716e-163,38.230592,41.952267,0.000000,0.00000
3,sensor_14,0.131042,3.369093e-144,-34.303976,-36.283095,0.000000,0.00000
4,sensor_06,0.107125,2.169975e-96,33.193240,32.204642,0.000000,0.00000
5,harmonic_thd,0.091854,6.171524e-71,3.205092,3.480921,0.000000,0.00000
6,grid_freq,0.083271,2.342515e-58,49.900486,49.924043,0.000000,0.00000
7,load_factor,0.082083,1.028762e-56,6.566271,7.253202,0.000000,0.00000
8,humidity,0.079465,5.715894e-49,47.733197,50.443544,7.891667,8.04375
9,sensor_09,0.075188,1.208656e-47,107.857737,104.272153,0.000000,0.00000


## 4. Exact 5-fold OOF baseline

This is the main checkpoint. The same fold-built 58-feature tables are reused for CatBoost and LightGBM, avoiding duplicate preprocessing work.

Expected reference results from the successful experiment:

- CatBoost OOF: approximately `0.190136`
- LightGBM OOF: approximately `0.196971`
- fixed 80/20 blend OOF: approximately `0.189636`


In [11]:
fixed_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

fixed_folds = list(fixed_cv.split(X_raw))

print("Number of fixed folds:", len(fixed_folds))


Number of fixed folds: 5


In [12]:
def create_cat_model():
    return CatBoostRegressor(
        iterations=2000,
        depth=6,
        learning_rate=0.03,
        l2_leaf_reg=3.0,
        loss_function="RMSE",
        random_seed=SEED,
        verbose=0,
        early_stopping_rounds=100,
    )


def create_lgb_model():
    return LGBMRegressor(
        objective="regression",
        n_estimators=4000,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=30,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    )


In [13]:
cat_oof_log = np.zeros(len(X_raw))
lgb_oof_log = np.zeros(len(X_raw))

cat_fold_scores = []
lgb_fold_scores = []

cat_best_iterations = []
lgb_best_iterations = []

for fold, (train_idx, valid_idx) in enumerate(
    fixed_folds,
    start=1,
):
    print("\n" + "=" * 60)
    print(f"BASELINE FOLD {fold}")
    print("=" * 60)

    X_train_fold_raw = X_raw.iloc[train_idx].copy()
    X_valid_fold_raw = X_raw.iloc[valid_idx].copy()

    y_train_fold = y_log.iloc[train_idx]
    y_valid_fold = y_log.iloc[valid_idx]

    (
        X_train_fold,
        X_valid_fold,
        _,
        _,
    ) = build_fold_features(
        X_train_fold_raw,
        X_valid_fold_raw,
    )

    print("Train shape:", X_train_fold.shape)
    print("Validation shape:", X_valid_fold.shape)

    # CatBoost
    cat_model = create_cat_model()

    cat_model.fit(
        X_train_fold,
        y_train_fold,
        eval_set=(X_valid_fold, y_valid_fold),
        use_best_model=True,
        verbose=False,
    )

    cat_fold_log = cat_model.predict(X_valid_fold)
    cat_oof_log[valid_idx] = cat_fold_log

    cat_fold_pred = np.clip(np.expm1(cat_fold_log), 0, None)
    fold_actual = np.expm1(y_valid_fold)

    cat_score = calculate_rmsle(
        fold_actual,
        cat_fold_pred,
    )

    cat_fold_scores.append(cat_score)
    cat_best_iterations.append(cat_model.get_best_iteration())

    # LightGBM
    lgb_model = create_lgb_model()

    lgb_model.fit(
        X_train_fold,
        y_train_fold,
        eval_set=[(X_valid_fold, y_valid_fold)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    lgb_fold_log = lgb_model.predict(
        X_valid_fold,
        num_iteration=lgb_model.best_iteration_,
    )
    lgb_oof_log[valid_idx] = lgb_fold_log

    lgb_fold_pred = np.clip(np.expm1(lgb_fold_log), 0, None)

    lgb_score = calculate_rmsle(
        fold_actual,
        lgb_fold_pred,
    )

    lgb_fold_scores.append(lgb_score)
    lgb_best_iterations.append(lgb_model.best_iteration_)

    print("CatBoost RMSLE:", cat_score)
    print("CatBoost best iteration:", cat_model.get_best_iteration())
    print("LightGBM RMSLE:", lgb_score)
    print("LightGBM best iteration:", lgb_model.best_iteration_)



BASELINE FOLD 1
Train shape: (19200, 58)
Validation shape: (4800, 58)
CatBoost RMSLE: 0.19377168144710952
CatBoost best iteration: 1930
LightGBM RMSLE: 0.2023844467128128
LightGBM best iteration: 1319

BASELINE FOLD 2
Train shape: (19200, 58)
Validation shape: (4800, 58)
CatBoost RMSLE: 0.18855780787060383
CatBoost best iteration: 1557
LightGBM RMSLE: 0.19472330708843621
LightGBM best iteration: 1483

BASELINE FOLD 3
Train shape: (19200, 58)
Validation shape: (4800, 58)
CatBoost RMSLE: 0.1884482802589682
CatBoost best iteration: 1520
LightGBM RMSLE: 0.19328720840093425
LightGBM best iteration: 1749

BASELINE FOLD 4
Train shape: (19200, 58)
Validation shape: (4800, 58)
CatBoost RMSLE: 0.1879147839158609
CatBoost best iteration: 1593
LightGBM RMSLE: 0.19577980129277028
LightGBM best iteration: 1396

BASELINE FOLD 5
Train shape: (19200, 58)
Validation shape: (4800, 58)
CatBoost RMSLE: 0.19191749463866725
CatBoost best iteration: 1567
LightGBM RMSLE: 0.19855074861966412
LightGBM best iter

In [14]:
cat_oof_pred = np.clip(np.expm1(cat_oof_log), 0, None)
lgb_oof_pred = np.clip(np.expm1(lgb_oof_log), 0, None)

blend_oof_log = (
    0.80 * cat_oof_log
    + 0.20 * lgb_oof_log
)

blend_oof_pred = np.clip(
    np.expm1(blend_oof_log),
    0,
    None,
)

cat_global_oof = calculate_rmsle(y_raw, cat_oof_pred)
lgb_global_oof = calculate_rmsle(y_raw, lgb_oof_pred)
blend_global_oof = calculate_rmsle(y_raw, blend_oof_pred)

print("CatBoost global OOF:", cat_global_oof)
print("LightGBM global OOF:", lgb_global_oof)
print("80/20 blend global OOF:", blend_global_oof)

print("\nCatBoost fold scores:", cat_fold_scores)
print("LightGBM fold scores:", lgb_fold_scores)

print("\nCatBoost best iterations:", cat_best_iterations)
print("LightGBM best iterations:", lgb_best_iterations)


CatBoost global OOF: 0.1901360336889869
LightGBM global OOF: 0.1969714222407914
80/20 blend global OOF: 0.18963602921932898

CatBoost fold scores: [0.19377168144710952, 0.18855780787060383, 0.1884482802589682, 0.1879147839158609, 0.19191749463866725]
LightGBM fold scores: [0.2023844467128128, 0.19472330708843621, 0.19328720840093425, 0.19577980129277028, 0.19855074861966412]

CatBoost best iterations: [1930, 1557, 1520, 1593, 1567]
LightGBM best iterations: [1319, 1483, 1749, 1396, 1032]


## 5. Robustness checkpoints

These metrics answer different questions:

- **Global OOF:** average performance across all training rows.
- **Test-like OOF:** performance on the 20% of training rows most similar to test.
- **Young-hot OOF:** performance in the known difficult region.
- **Fold spread:** stability across folds.

A candidate is worth considering only when its global OOF is competitive and it improves or preserves the test-like checkpoint.

In [15]:
young_threshold = train["asset_age"].quantile(0.25)
hot_threshold = train["core_temp"].quantile(0.75)

young_hot_mask = (
    (train["asset_age"] <= young_threshold)
    & (train["core_temp"] >= hot_threshold)
)

checkpoint_rows = []

for model_name, prediction in {
    "CatBoost": cat_oof_pred,
    "LightGBM": lgb_oof_pred,
    "Blend_80_20": blend_oof_pred,
}.items():
    checkpoint_rows.append({
        "model": model_name,
        "global_oof": calculate_rmsle(
            y_raw,
            prediction,
        ),
        "test_like_oof": calculate_rmsle(
            y_raw[very_test_like_mask],
            prediction[very_test_like_mask],
        ),
        "young_hot_oof": calculate_rmsle(
            y_raw[young_hot_mask],
            prediction[young_hot_mask],
        ),
    })

checkpoint_table = pd.DataFrame(checkpoint_rows)
display(checkpoint_table)


,model,global_oof,test_like_oof,young_hot_oof
0,CatBoost,0.190136,0.231128,0.288159
1,LightGBM,0.196971,0.238572,0.296824
2,Blend_80_20,0.189636,0.230337,0.287495


In [16]:
actual_log = np.log1p(y_raw.values)

cat_error = cat_oof_log - actual_log
lgb_error = lgb_oof_log - actual_log

error_correlation = np.corrcoef(
    cat_error,
    lgb_error,
)[0, 1]

fold_stability = pd.DataFrame({
    "fold": np.arange(1, 6),
    "cat_rmsle": cat_fold_scores,
    "lgb_rmsle": lgb_fold_scores,
})

print("CatBoost/LightGBM OOF error correlation:", error_correlation)
display(fold_stability)


CatBoost/LightGBM OOF error correlation: 0.9406231398448397


,fold,cat_rmsle,lgb_rmsle
0,1,0.193772,0.202384
1,2,0.188558,0.194723
2,3,0.188448,0.193287
3,4,0.187915,0.195780
4,5,0.191917,0.198551


### Optional blend diagnostic

This does not automatically replace the proven 80/20 blend. It shows whether a candidate model changes the preferred blend region.

In [17]:
blend_diagnostic = []

for cat_weight in np.arange(0.50, 1.01, 0.05):
    lgb_weight = 1.0 - cat_weight

    candidate_log = (
        cat_weight * cat_oof_log
        + lgb_weight * lgb_oof_log
    )

    candidate_pred = np.clip(
        np.expm1(candidate_log),
        0,
        None,
    )

    blend_diagnostic.append({
        "cat_weight": round(float(cat_weight), 2),
        "lgb_weight": round(float(lgb_weight), 2),
        "global_oof": calculate_rmsle(
            y_raw,
            candidate_pred,
        ),
        "test_like_oof": calculate_rmsle(
            y_raw[very_test_like_mask],
            candidate_pred[very_test_like_mask],
        ),
        "young_hot_oof": calculate_rmsle(
            y_raw[young_hot_mask],
            candidate_pred[young_hot_mask],
        ),
    })

blend_diagnostic = pd.DataFrame(blend_diagnostic)
display(blend_diagnostic.sort_values("global_oof"))


,cat_weight,lgb_weight,global_oof,test_like_oof,young_hot_oof
6,0.80,0.20,0.189636,0.230337,0.287495
5,0.75,0.25,0.189659,0.230320,0.287518
7,0.85,0.15,0.189672,0.230427,0.287547
4,0.70,0.30,0.189741,0.230374,0.287617
8,0.90,0.10,0.189768,0.230589,0.287675
3,0.65,0.35,0.189882,0.230501,0.287792
9,0.95,0.05,0.189922,0.230822,0.287880
2,0.60,0.40,0.190083,0.230700,0.288043
10,1.00,-0.00,0.190136,0.231128,0.288159
1,0.55,0.45,0.190342,0.230970,0.288369


## 6. Save the baseline checkpoint

This prevents future experiments from losing or silently replacing the proven baseline.

In [24]:
CHECKPOINT_DIR = (
    "/content/drive/MyDrive/"
    "EDI_Hackathon/baseline_checkpoint"
)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

np.save(
    os.path.join(CHECKPOINT_DIR, "cat_oof_log.npy"),
    cat_oof_log,
)
np.save(
    os.path.join(CHECKPOINT_DIR, "lgb_oof_log.npy"),
    lgb_oof_log,
)
np.save(
    os.path.join(CHECKPOINT_DIR, "blend_oof_log.npy"),
    blend_oof_log,
)
np.save(
    os.path.join(CHECKPOINT_DIR, "train_test_likeness.npy"),
    train_test_likeness,
)
np.save(
    os.path.join(CHECKPOINT_DIR, "very_test_like_mask.npy"),
    very_test_like_mask,
)

checkpoint_table.to_csv(
    os.path.join(CHECKPOINT_DIR, "checkpoint_scores.csv"),
    index=False,
)

baseline_metadata = {
    "seed": SEED,
    "feature_count": 58,
    "cat_global_oof": cat_global_oof,
    "lgb_global_oof": lgb_global_oof,
    "blend_global_oof": blend_global_oof,
    "shift_auc": float(shift_auc),
    "cat_best_iterations": [
        int(value) for value in cat_best_iterations
    ],
    "lgb_best_iterations": [
        int(value) for value in lgb_best_iterations
    ],
    "proven_kaggle_score": 0.20500,
    "proven_cat_final_iterations": 1840,
    "proven_lgb_final_iterations": 1396,
    "blend_cat_weight": 0.80,
    "blend_lgb_weight": 0.20,
}

with open(
    os.path.join(CHECKPOINT_DIR, "baseline_metadata.json"),
    "w",
) as file:
    json.dump(baseline_metadata, file, indent=2)

print("Saved checkpoint files to:", CHECKPOINT_DIR)


Saved checkpoint files to: /content/drive/MyDrive/EDI_Hackathon/baseline_checkpoint


In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
import shutil

shutil.make_archive(
    "/content/baseline_checkpoint",
    "zip",
    CHECKPOINT_DIR
)

print("ZIP created")

ZIP created


## 7. Candidate experiment scoreboard

For every future experiment:

1. produce fully out-of-fold log predictions;
2. evaluate them with this function;
3. compare against `Blend_80_20`;
4. reject candidates that improve only one tiny metric while harming robustness.

Do not create a submission from a candidate before this comparison.

In [19]:
BASELINE_GLOBAL = blend_global_oof
BASELINE_TEST_LIKE = calculate_rmsle(
    y_raw[very_test_like_mask],
    blend_oof_pred[very_test_like_mask],
)
BASELINE_YOUNG_HOT = calculate_rmsle(
    y_raw[young_hot_mask],
    blend_oof_pred[young_hot_mask],
)


def evaluate_candidate_oof(
    candidate_name,
    candidate_oof_log,
):
    candidate_oof_log = np.asarray(candidate_oof_log)

    if candidate_oof_log.shape != (len(train),):
        raise ValueError(
            f"Expected shape {(len(train),)}, "
            f"received {candidate_oof_log.shape}"
        )

    candidate_pred = np.clip(
        np.expm1(candidate_oof_log),
        0,
        None,
    )

    result = pd.DataFrame([{
        "candidate": candidate_name,
        "global_oof": calculate_rmsle(
            y_raw,
            candidate_pred,
        ),
        "global_change": calculate_rmsle(
            y_raw,
            candidate_pred,
        ) - BASELINE_GLOBAL,
        "test_like_oof": calculate_rmsle(
            y_raw[very_test_like_mask],
            candidate_pred[very_test_like_mask],
        ),
        "test_like_change": calculate_rmsle(
            y_raw[very_test_like_mask],
            candidate_pred[very_test_like_mask],
        ) - BASELINE_TEST_LIKE,
        "young_hot_oof": calculate_rmsle(
            y_raw[young_hot_mask],
            candidate_pred[young_hot_mask],
        ),
        "young_hot_change": calculate_rmsle(
            y_raw[young_hot_mask],
            candidate_pred[young_hot_mask],
        ) - BASELINE_YOUNG_HOT,
    }])

    return result


# Example:
# candidate_report = evaluate_candidate_oof(
#     "CatBoost_depth_5",
#     candidate_oof_log,
# )
# display(candidate_report)


### Practical decision rule

Lower RMSLE is better.

A candidate is **promising** when:

- global OOF improves by a non-trivial amount, or stays almost unchanged;
- test-like OOF clearly improves;
- no severe deterioration appears in young-hot or individual folds;
- the result comes from the same fixed folds and fold-safe preprocessing.

A difference around `0.00001` is normally too small to trust. Keep the exact numerical report rather than declaring a win from rounding.

## 8. Exact final model and submission

Run this only for the proven baseline or for a candidate that passed the checkpoints.

The fixed iterations below reproduce the successful 0.20500 pipeline:

- CatBoost: `1840`
- LightGBM: `1396`
- log-space blend: `80% / 20%`


In [20]:
X_full_train_thermal, X_full_test_thermal, final_temperature_model = (
    add_temperature_residual_features(
        X_raw.copy(),
        X_test_raw.copy(),
    )
)

X_full_train, X_full_test, full_medians = preprocess_fold(
    X_full_train_thermal,
    X_full_test_thermal,
)

X_full_train = add_engineered_features_stress(X_full_train)
X_full_test = add_engineered_features_stress(X_full_test)

assert list(X_full_train.columns) == list(X_full_test.columns)
assert X_full_train.shape == (24000, 58)
assert X_full_test.shape == (16000, 58)
assert X_full_train.isna().sum().sum() == 0
assert X_full_test.isna().sum().sum() == 0

print("Final train features:", X_full_train.shape)
print("Final test features:", X_full_test.shape)


Final train features: (24000, 58)
Final test features: (16000, 58)


In [21]:
FINAL_CAT_ITERATIONS = 1840
FINAL_LGB_ITERATIONS = 1396

final_cat_model = CatBoostRegressor(
    iterations=FINAL_CAT_ITERATIONS,
    depth=6,
    learning_rate=0.03,
    l2_leaf_reg=3.0,
    loss_function="RMSE",
    random_seed=SEED,
    verbose=100,
)

final_cat_model.fit(
    X_full_train,
    y_log,
)

cat_test_log = final_cat_model.predict(X_full_test)


0:	learn: 0.9283741	total: 27.7ms	remaining: 50.9s
100:	learn: 0.2744276	total: 2.42s	remaining: 41.7s
200:	learn: 0.2143198	total: 6.36s	remaining: 51.9s
300:	learn: 0.1983686	total: 9.02s	remaining: 46.1s
400:	learn: 0.1901752	total: 11.1s	remaining: 39.9s
500:	learn: 0.1843760	total: 13.2s	remaining: 35.3s
600:	learn: 0.1801740	total: 15.3s	remaining: 31.5s
700:	learn: 0.1767324	total: 17.4s	remaining: 28.3s
800:	learn: 0.1737308	total: 21.3s	remaining: 27.6s
900:	learn: 0.1710598	total: 24.1s	remaining: 25.1s
1000:	learn: 0.1685288	total: 26.2s	remaining: 22s
1100:	learn: 0.1661713	total: 28.3s	remaining: 19s
1200:	learn: 0.1638472	total: 30.4s	remaining: 16.2s
1300:	learn: 0.1616742	total: 32.6s	remaining: 13.5s
1400:	learn: 0.1595701	total: 36.1s	remaining: 11.3s
1500:	learn: 0.1575931	total: 39.3s	remaining: 8.87s
1600:	learn: 0.1555828	total: 41.4s	remaining: 6.18s
1700:	learn: 0.1537211	total: 43.5s	remaining: 3.56s
1800:	learn: 0.1518733	total: 45.7s	remaining: 990ms
1839:	le

In [22]:
final_lgb_model = LGBMRegressor(
    objective="regression",
    n_estimators=FINAL_LGB_ITERATIONS,
    learning_rate=0.02,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

final_lgb_model.fit(
    X_full_train,
    y_log,
)

lgb_test_log = final_lgb_model.predict(X_full_test)


In [23]:
final_blend_test_log = (
    0.80 * cat_test_log
    + 0.20 * lgb_test_log
)

final_blend_test_pred = np.clip(
    np.expm1(final_blend_test_log),
    0,
    None,
)

submission = pd.DataFrame({
    "id": test["id"].values,
    "edi": final_blend_test_pred,
})

submission.to_csv(
    "submission_clean_best_020500.csv",
    index=False,
)

print("Submission shape:", submission.shape)
print("Missing:", submission["edi"].isna().sum())
print("Infinite:", np.isinf(submission["edi"]).sum())
print("Duplicate IDs:", submission["id"].duplicated().sum())
print("Minimum prediction:", submission["edi"].min())
print("Maximum prediction:", submission["edi"].max())

display(submission.head())

assert submission.shape == (16000, 2)
assert submission["edi"].isna().sum() == 0
assert np.isfinite(submission["edi"]).all()
assert submission["id"].equals(test["id"])


Submission shape: (16000, 2)
Missing: 0
Infinite: 0
Duplicate IDs: 0
Minimum prediction: 3.9652839915435703
Maximum prediction: 737.3295358422752


,id,edi
0,24000,46.171586
1,24001,37.690626
2,24002,51.375100
3,24003,78.267724
4,24004,31.575184


## 9. What to work on next

The clean notebook establishes the compass. The next experiments should be controlled and limited to one hypothesis at a time.

Recommended order:

1. CatBoost robustness: depth and regularization.
2. Re-evaluate blend weights using both global and test-like checkpoints.
3. Improve the thermal residual predictor, because thermal residuals already helped the leaderboard.
4. Carefully test pseudo-labeling only after the validation checkpoints are stable.
5. Consider external or synthetic data only when its physical compatibility and labels are defensible.

Do not return to broad GMM clustering, manual regional corrections, or dozens of unmotivated interactions unless new evidence supports them.


In [27]:
# ============================================================
# EXPERIMENT 01 — CATBOOST ROBUSTNESS
# Prepare the same fold-safe 58-feature data once
# ============================================================

experiment_fold_data = []

for fold, (train_idx, valid_idx) in enumerate(
    fixed_folds,
    start=1
):
    print(f"Preparing fold {fold}...")

    X_train_raw_fold = X_raw.iloc[train_idx].copy()
    X_valid_raw_fold = X_raw.iloc[valid_idx].copy()

    (
        X_train_fold,
        X_valid_fold,
        _,
        _
    ) = build_fold_features(
        X_train_raw_fold,
        X_valid_raw_fold
    )

    experiment_fold_data.append({
        "fold": fold,
        "train_idx": train_idx,
        "valid_idx": valid_idx,
        "X_train": X_train_fold,
        "X_valid": X_valid_fold,
        "y_train": y_log.iloc[train_idx],
        "y_valid": y_log.iloc[valid_idx]
    })

    print(
        f"Fold {fold} ready:",
        X_train_fold.shape,
        X_valid_fold.shape
    )

print("\nAll experiment folds are ready.")

Preparing fold 1...
Fold 1 ready: (19200, 58) (4800, 58)
Preparing fold 2...
Fold 2 ready: (19200, 58) (4800, 58)
Preparing fold 3...
Fold 3 ready: (19200, 58) (4800, 58)
Preparing fold 4...
Fold 4 ready: (19200, 58) (4800, 58)
Preparing fold 5...
Fold 5 ready: (19200, 58) (4800, 58)

All experiment folds are ready.


In [28]:
catboost_candidates = {
    "cat_depth5_l2_10": {
        "depth": 5,
        "l2_leaf_reg": 10.0
    },

    "cat_depth6_l2_10": {
        "depth": 6,
        "l2_leaf_reg": 10.0
    },

    "cat_depth7_l2_10": {
        "depth": 7,
        "l2_leaf_reg": 10.0
    }
}

catboost_candidates

{'cat_depth5_l2_10': {'depth': 5, 'l2_leaf_reg': 10.0},
 'cat_depth6_l2_10': {'depth': 6, 'l2_leaf_reg': 10.0},
 'cat_depth7_l2_10': {'depth': 7, 'l2_leaf_reg': 10.0}}

In [29]:
candidate_cat_oof_logs = {}
candidate_cat_fold_scores = {}
candidate_cat_best_iterations = {}

for candidate_name, parameters in catboost_candidates.items():

    print("\n" + "#" * 70)
    print("CANDIDATE:", candidate_name)
    print(parameters)
    print("#" * 70)

    candidate_oof_log = np.zeros(len(X_raw))
    fold_scores = []
    best_iterations = []

    for fold_data in experiment_fold_data:

        fold = fold_data["fold"]
        train_idx = fold_data["train_idx"]
        valid_idx = fold_data["valid_idx"]

        X_train_fold = fold_data["X_train"]
        X_valid_fold = fold_data["X_valid"]

        y_train_fold = fold_data["y_train"]
        y_valid_fold = fold_data["y_valid"]

        print(f"\nTraining fold {fold}...")

        candidate_model = CatBoostRegressor(
            iterations=2500,
            depth=parameters["depth"],
            learning_rate=0.03,
            l2_leaf_reg=parameters["l2_leaf_reg"],
            loss_function="RMSE",
            random_seed=SEED,
            verbose=0,
            early_stopping_rounds=100
        )

        candidate_model.fit(
            X_train_fold,
            y_train_fold,
            eval_set=(
                X_valid_fold,
                y_valid_fold
            ),
            use_best_model=True,
            verbose=False
        )

        fold_pred_log = candidate_model.predict(
            X_valid_fold
        )

        candidate_oof_log[valid_idx] = fold_pred_log

        fold_pred = np.clip(
            np.expm1(fold_pred_log),
            0,
            None
        )

        fold_actual = np.expm1(
            y_valid_fold
        )

        fold_score = calculate_rmsle(
            fold_actual,
            fold_pred
        )

        fold_scores.append(fold_score)
        best_iterations.append(
            candidate_model.get_best_iteration()
        )

        print(
            f"Fold {fold} RMSLE:",
            fold_score
        )

        print(
            f"Fold {fold} best iteration:",
            candidate_model.get_best_iteration()
        )

    candidate_cat_oof_logs[
        candidate_name
    ] = candidate_oof_log

    candidate_cat_fold_scores[
        candidate_name
    ] = fold_scores

    candidate_cat_best_iterations[
        candidate_name
    ] = best_iterations


######################################################################
CANDIDATE: cat_depth5_l2_10
{'depth': 5, 'l2_leaf_reg': 10.0}
######################################################################

Training fold 1...
Fold 1 RMSLE: 0.19296910302058892
Fold 1 best iteration: 2497

Training fold 2...
Fold 2 RMSLE: 0.18891343855531664
Fold 2 best iteration: 1819

Training fold 3...
Fold 3 RMSLE: 0.18966657769040351
Fold 3 best iteration: 1882

Training fold 4...
Fold 4 RMSLE: 0.18813611495215654
Fold 4 best iteration: 2463

Training fold 5...
Fold 5 RMSLE: 0.191745674496303
Fold 5 best iteration: 2493

######################################################################
CANDIDATE: cat_depth6_l2_10
{'depth': 6, 'l2_leaf_reg': 10.0}
######################################################################

Training fold 1...
Fold 1 RMSLE: 0.19422319039613
Fold 1 best iteration: 1672

Training fold 2...
Fold 2 RMSLE: 0.1877809043199515
Fold 2 best iteration: 1709

Training fold 3...
Fo

In [30]:
experiment_results = []

for candidate_name, candidate_cat_log in (
    candidate_cat_oof_logs.items()
):

    # Candidate CatBoost alone
    candidate_cat_pred = np.clip(
        np.expm1(candidate_cat_log),
        0,
        None
    )

    candidate_cat_global = calculate_rmsle(
        y_raw,
        candidate_cat_pred
    )

    candidate_cat_test_like = calculate_rmsle(
        y_raw[very_test_like_mask],
        candidate_cat_pred[very_test_like_mask]
    )

    candidate_cat_young_hot = calculate_rmsle(
        y_raw[young_hot_mask],
        candidate_cat_pred[young_hot_mask]
    )

    # Candidate CatBoost + original LightGBM
    candidate_blend_log = (
        0.80 * candidate_cat_log
        + 0.20 * lgb_oof_log
    )

    candidate_blend_pred = np.clip(
        np.expm1(candidate_blend_log),
        0,
        None
    )

    candidate_blend_global = calculate_rmsle(
        y_raw,
        candidate_blend_pred
    )

    candidate_blend_test_like = calculate_rmsle(
        y_raw[very_test_like_mask],
        candidate_blend_pred[very_test_like_mask]
    )

    candidate_blend_young_hot = calculate_rmsle(
        y_raw[young_hot_mask],
        candidate_blend_pred[young_hot_mask]
    )

    experiment_results.append({
        "candidate": candidate_name,

        "cat_global_oof": candidate_cat_global,
        "cat_test_like_oof": candidate_cat_test_like,
        "cat_young_hot_oof": candidate_cat_young_hot,

        "blend_global_oof": candidate_blend_global,
        "blend_global_change": (
            candidate_blend_global
            - BASELINE_GLOBAL
        ),

        "blend_test_like_oof": candidate_blend_test_like,
        "blend_test_like_change": (
            candidate_blend_test_like
            - BASELINE_TEST_LIKE
        ),

        "blend_young_hot_oof": candidate_blend_young_hot,
        "blend_young_hot_change": (
            candidate_blend_young_hot
            - BASELINE_YOUNG_HOT
        ),

        "mean_best_iteration": np.mean(
            candidate_cat_best_iterations[
                candidate_name
            ]
        )
    })

experiment_results_df = pd.DataFrame(
    experiment_results
)

display(
    experiment_results_df.sort_values(
        "blend_global_oof"
    )
)

,candidate,cat_global_oof,cat_test_like_oof,cat_young_hot_oof,blend_global_oof,blend_global_change,blend_test_like_oof,blend_test_like_change,blend_young_hot_oof,blend_young_hot_change,mean_best_iteration
2,cat_depth7_l2_10,0.189764,0.230381,0.287599,0.189418,-0.000218,0.229840,-0.000497,0.287083,-0.000412,1786.0
1,cat_depth6_l2_10,0.189985,0.230006,0.286540,0.189514,-0.000122,0.229411,-0.000926,0.286203,-0.001292,1932.0
0,cat_depth5_l2_10,0.190295,0.231475,0.287501,0.189676,0.000040,0.230487,0.000150,0.286958,-0.000537,2230.8


In [31]:
print("PROVEN BASELINE")
print("------------------------------")
print("Global OOF:    ", BASELINE_GLOBAL)
print("Test-like OOF: ", BASELINE_TEST_LIKE)
print("Young-hot OOF: ", BASELINE_YOUNG_HOT)

print("\nCANDIDATE RESULTS")
display(
    experiment_results_df[
        [
            "candidate",
            "blend_global_oof",
            "blend_global_change",
            "blend_test_like_oof",
            "blend_test_like_change",
            "blend_young_hot_oof",
            "blend_young_hot_change",
            "mean_best_iteration"
        ]
    ].sort_values(
        "blend_global_oof"
    )
)

PROVEN BASELINE
------------------------------
Global OOF:     0.18963602921932898
Test-like OOF:  0.23033713788172333
Young-hot OOF:  0.2874946673196195

CANDIDATE RESULTS


,candidate,blend_global_oof,blend_global_change,blend_test_like_oof,blend_test_like_change,blend_young_hot_oof,blend_young_hot_change,mean_best_iteration
2,cat_depth7_l2_10,0.189418,-0.000218,0.229840,-0.000497,0.287083,-0.000412,1786.0
1,cat_depth6_l2_10,0.189514,-0.000122,0.229411,-0.000926,0.286203,-0.001292,1932.0
0,cat_depth5_l2_10,0.189676,0.000040,0.230487,0.000150,0.286958,-0.000537,2230.8


In [32]:
cat6_log = candidate_cat_oof_logs[
    "cat_depth6_l2_10"
]

cat7_log = candidate_cat_oof_logs[
    "cat_depth7_l2_10"
]

ensemble_search_results = []

for lgb_weight in [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30
]:
    total_cat_weight = 1.0 - lgb_weight

    for cat7_share in np.arange(
        0.0,
        1.01,
        0.10
    ):
        cat6_share = 1.0 - cat7_share

        mixed_cat_log = (
            cat6_share * cat6_log
            + cat7_share * cat7_log
        )

        final_candidate_log = (
            total_cat_weight * mixed_cat_log
            + lgb_weight * lgb_oof_log
        )

        final_candidate_pred = np.clip(
            np.expm1(final_candidate_log),
            0,
            None
        )

        global_score = calculate_rmsle(
            y_raw,
            final_candidate_pred
        )

        test_like_score = calculate_rmsle(
            y_raw[very_test_like_mask],
            final_candidate_pred[
                very_test_like_mask
            ]
        )

        young_hot_score = calculate_rmsle(
            y_raw[young_hot_mask],
            final_candidate_pred[
                young_hot_mask
            ]
        )

        ensemble_search_results.append({
            "cat6_share": cat6_share,
            "cat7_share": cat7_share,
            "total_cat_weight": total_cat_weight,
            "lgb_weight": lgb_weight,

            "global_oof": global_score,
            "global_change": (
                global_score - BASELINE_GLOBAL
            ),

            "test_like_oof": test_like_score,
            "test_like_change": (
                test_like_score
                - BASELINE_TEST_LIKE
            ),

            "young_hot_oof": young_hot_score,
            "young_hot_change": (
                young_hot_score
                - BASELINE_YOUNG_HOT
            )
        })

ensemble_search_df = pd.DataFrame(
    ensemble_search_results
)

In [33]:
successful_ensembles = ensemble_search_df[
    (ensemble_search_df["global_change"] < 0)
    & (ensemble_search_df["test_like_change"] < 0)
    & (ensemble_search_df["young_hot_change"] < 0)
].copy()

display(
    successful_ensembles.sort_values(
        "global_oof"
    ).head(15)
)

,cat6_share,cat7_share,total_cat_weight,lgb_weight,global_oof,global_change,test_like_oof,test_like_change,young_hot_oof,young_hot_change
5,0.5,0.5,0.90,0.10,0.188875,-0.000761,0.228993,-0.001344,0.285889,-0.001606
16,0.5,0.5,0.85,0.15,0.188875,-0.000761,0.228970,-0.001367,0.285908,-0.001587
6,0.4,0.6,0.90,0.10,0.188887,-0.000749,0.229065,-0.001272,0.286019,-0.001476
17,0.4,0.6,0.85,0.15,0.188887,-0.000749,0.229040,-0.001297,0.286030,-0.001465
15,0.6,0.4,0.85,0.15,0.188912,-0.000724,0.228955,-0.001382,0.285844,-0.001651
4,0.6,0.4,0.90,0.10,0.188918,-0.000718,0.228983,-0.001355,0.285824,-0.001671
27,0.5,0.5,0.80,0.20,0.188929,-0.000707,0.229013,-0.001324,0.285997,-0.001497
28,0.4,0.6,0.80,0.20,0.188941,-0.000695,0.229081,-0.001256,0.286111,-0.001383
18,0.3,0.7,0.85,0.15,0.188948,-0.000689,0.229165,-0.001172,0.286210,-0.001285
7,0.3,0.7,0.90,0.10,0.188953,-0.000683,0.229199,-0.001138,0.286214,-0.001280


In [34]:
display(
    successful_ensembles.sort_values(
        "test_like_oof"
    ).head(15)
)

,cat6_share,cat7_share,total_cat_weight,lgb_weight,global_oof,global_change,test_like_oof,test_like_change,young_hot_oof,young_hot_change
15,0.6,0.4,0.85,0.15,0.188912,-0.000724,0.228955,-0.001382,0.285844,-0.001651
16,0.5,0.5,0.85,0.15,0.188875,-0.000761,0.228970,-0.001367,0.285908,-0.001587
4,0.6,0.4,0.90,0.10,0.188918,-0.000718,0.228983,-0.001355,0.285824,-0.001671
5,0.5,0.5,0.90,0.10,0.188875,-0.000761,0.228993,-0.001344,0.285889,-0.001606
26,0.6,0.4,0.80,0.20,0.188960,-0.000676,0.228995,-0.001342,0.285935,-0.001560
14,0.7,0.3,0.85,0.15,0.188997,-0.000639,0.228996,-0.001341,0.285839,-0.001656
27,0.5,0.5,0.80,0.20,0.188929,-0.000707,0.229013,-0.001324,0.285997,-0.001497
25,0.7,0.3,0.80,0.20,0.189035,-0.000602,0.229026,-0.001312,0.285925,-0.001570
3,0.7,0.3,0.90,0.10,0.189015,-0.000621,0.229035,-0.001303,0.285825,-0.001670
17,0.4,0.6,0.85,0.15,0.188887,-0.000749,0.229040,-0.001297,0.286030,-0.001465


In [35]:
assert X_full_train.shape == (24000, 58)
assert X_full_test.shape == (16000, 58)

In [36]:
final_cat6_model = CatBoostRegressor(
    iterations=1932,
    depth=6,
    learning_rate=0.03,
    l2_leaf_reg=10.0,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

final_cat6_model.fit(
    X_full_train,
    y_log
)

final_cat6_test_log = final_cat6_model.predict(
    X_full_test
)

print("Depth-6 predictions:", len(final_cat6_test_log))
print("Missing:", np.isnan(final_cat6_test_log).sum())
print("Infinite:", np.isinf(final_cat6_test_log).sum())

0:	learn: 0.9285949	total: 61ms	remaining: 1m 57s
100:	learn: 0.2782714	total: 3.96s	remaining: 1m 11s
200:	learn: 0.2167902	total: 7.91s	remaining: 1m 8s
300:	learn: 0.2007968	total: 10.8s	remaining: 58.6s
400:	learn: 0.1925137	total: 12.9s	remaining: 49.2s
500:	learn: 0.1866445	total: 16.2s	remaining: 46.2s
600:	learn: 0.1826126	total: 19.9s	remaining: 44.1s
700:	learn: 0.1793976	total: 23.9s	remaining: 41.9s
800:	learn: 0.1767135	total: 26.6s	remaining: 37.6s
900:	learn: 0.1741965	total: 28.8s	remaining: 32.9s
1000:	learn: 0.1719747	total: 30.9s	remaining: 28.7s
1100:	learn: 0.1699479	total: 33s	remaining: 24.9s
1200:	learn: 0.1680810	total: 35.1s	remaining: 21.3s
1300:	learn: 0.1661140	total: 38.7s	remaining: 18.8s
1400:	learn: 0.1642830	total: 41.9s	remaining: 15.9s
1500:	learn: 0.1624981	total: 44s	remaining: 12.6s
1600:	learn: 0.1607496	total: 46.1s	remaining: 9.54s
1700:	learn: 0.1590336	total: 48.3s	remaining: 6.56s
1800:	learn: 0.1573848	total: 50.4s	remaining: 3.67s
1900:	le

In [39]:
final_cat7_model = CatBoostRegressor(
    iterations=1786,
    depth=7,
    learning_rate=0.03,
    l2_leaf_reg=10.0,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

final_cat7_model.fit(
    X_full_train,
    y_log
)

final_cat7_test_log = final_cat7_model.predict(
    X_full_test
)

print("Depth-7 predictions:", len(final_cat7_test_log))
print("Missing:", np.isnan(final_cat7_test_log).sum())
print("Infinite:", np.isinf(final_cat7_test_log).sum())

0:	learn: 0.9285539	total: 181ms	remaining: 5m 23s
100:	learn: 0.2642407	total: 6.15s	remaining: 1m 42s
200:	learn: 0.2092911	total: 9.55s	remaining: 1m 15s
300:	learn: 0.1949304	total: 14.3s	remaining: 1m 10s
400:	learn: 0.1873549	total: 18.8s	remaining: 1m 4s
500:	learn: 0.1818938	total: 22.1s	remaining: 56.7s
600:	learn: 0.1778148	total: 26.3s	remaining: 51.9s
700:	learn: 0.1743965	total: 32.7s	remaining: 50.5s
800:	learn: 0.1713896	total: 36s	remaining: 44.3s
900:	learn: 0.1684374	total: 39.3s	remaining: 38.6s
1000:	learn: 0.1656934	total: 42.8s	remaining: 33.6s
1100:	learn: 0.1630573	total: 48.6s	remaining: 30.2s
1200:	learn: 0.1606449	total: 52s	remaining: 25.3s
1300:	learn: 0.1581837	total: 55.4s	remaining: 20.6s
1400:	learn: 0.1559348	total: 59.2s	remaining: 16.3s
1500:	learn: 0.1536680	total: 1m 4s	remaining: 12.3s
1600:	learn: 0.1515017	total: 1m 8s	remaining: 7.87s
1700:	learn: 0.1493480	total: 1m 11s	remaining: 3.57s
1785:	learn: 0.1475149	total: 1m 14s	remaining: 0us
Depth

In [40]:
final_lgb_model = LGBMRegressor(
    objective="regression",
    n_estimators=1396,
    learning_rate=0.02,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,

    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

final_lgb_model.fit(
    X_full_train,
    y_log
)

final_lgb_test_log = final_lgb_model.predict(
    X_full_test
)

print("LightGBM predictions:", len(final_lgb_test_log))
print("Missing:", np.isnan(final_lgb_test_log).sum())
print("Infinite:", np.isinf(final_lgb_test_log).sum())

LightGBM predictions: 16000
Missing: 0
Infinite: 0


In [41]:
final_ensemble_test_log = (
    0.425 * final_cat6_test_log
    + 0.425 * final_cat7_test_log
    + 0.150 * final_lgb_test_log
)

final_ensemble_test_pred = np.clip(
    np.expm1(final_ensemble_test_log),
    0,
    None
)

submission_cat6_cat7_lgb = pd.DataFrame({
    "id": test["id"].values,
    "edi": final_ensemble_test_pred
})

submission_cat6_cat7_lgb.to_csv(
    "submission_cat6_425_cat7_425_lgb15.csv",
    index=False
)

print("Submission shape:", submission_cat6_cat7_lgb.shape)
print(
    "Missing predictions:",
    submission_cat6_cat7_lgb["edi"].isna().sum()
)
print(
    "Infinite predictions:",
    np.isinf(submission_cat6_cat7_lgb["edi"]).sum()
)
print(
    "Duplicate IDs:",
    submission_cat6_cat7_lgb["id"].duplicated().sum()
)
print(
    "Minimum prediction:",
    submission_cat6_cat7_lgb["edi"].min()
)
print(
    "Maximum prediction:",
    submission_cat6_cat7_lgb["edi"].max()
)

display(submission_cat6_cat7_lgb.head())

assert submission_cat6_cat7_lgb.shape == (16000, 2)
assert submission_cat6_cat7_lgb["edi"].isna().sum() == 0
assert np.isfinite(
    submission_cat6_cat7_lgb["edi"]
).all()
assert submission_cat6_cat7_lgb["id"].equals(
    test["id"]
)

Submission shape: (16000, 2)
Missing predictions: 0
Infinite predictions: 0
Duplicate IDs: 0
Minimum prediction: 4.0400348869007665
Maximum prediction: 721.3678723752075


,id,edi
0,24000,45.593473
1,24001,37.810524
2,24002,52.495806
3,24003,79.523642
4,24004,32.364284


In [42]:
best_new_oof_log = (
    0.425 * cat6_log
    + 0.425 * cat7_log
    + 0.150 * lgb_oof_log
)

np.save(
    os.path.join(
        CHECKPOINT_DIR,
        "best_cat6_cat7_lgb_oof_log.npy"
    ),
    best_new_oof_log
)

experiment_results_df.to_csv(
    os.path.join(
        CHECKPOINT_DIR,
        "catboost_robustness_results.csv"
    ),
    index=False
)

ensemble_search_df.to_csv(
    os.path.join(
        CHECKPOINT_DIR,
        "ensemble_weight_search.csv"
    ),
    index=False
)

print("New experiment checkpoints saved.")

New experiment checkpoints saved.


In [43]:
old_new_blend_results = []

for new_weight in np.arange(0.0, 1.01, 0.05):
    old_weight = 1.0 - new_weight

    mixed_oof_log = (
        old_weight * blend_oof_log
        + new_weight * best_new_oof_log
    )

    mixed_oof_pred = np.clip(
        np.expm1(mixed_oof_log),
        0,
        None
    )

    global_score = calculate_rmsle(
        y_raw,
        mixed_oof_pred
    )

    test_like_score = calculate_rmsle(
        y_raw[very_test_like_mask],
        mixed_oof_pred[very_test_like_mask]
    )

    young_hot_score = calculate_rmsle(
        y_raw[young_hot_mask],
        mixed_oof_pred[young_hot_mask]
    )

    old_new_blend_results.append({
        "old_weight": old_weight,
        "new_weight": new_weight,
        "global_oof": global_score,
        "test_like_oof": test_like_score,
        "young_hot_oof": young_hot_score
    })

old_new_blend_df = pd.DataFrame(
    old_new_blend_results
)

display(
    old_new_blend_df.sort_values(
        "global_oof"
    ).head(10)
)

,old_weight,new_weight,global_oof,test_like_oof,young_hot_oof
16,0.20,0.80,0.188834,0.228993,0.285971
17,0.15,0.85,0.188835,0.228976,0.285943
15,0.25,0.75,0.188839,0.229019,0.286006
18,0.10,0.90,0.188842,0.228966,0.285923
14,0.30,0.70,0.188850,0.229052,0.286050
19,0.05,0.95,0.188856,0.228964,0.285911
13,0.35,0.65,0.188867,0.229093,0.286101
20,0.00,1.00,0.188875,0.228970,0.285908
12,0.40,0.60,0.188890,0.229142,0.286161
11,0.45,0.55,0.188919,0.229199,0.286228


In [46]:
old_new_20_80_test_log = (
    0.20 * final_blend_test_log
    + 0.80 * final_ensemble_test_log
)

old_new_20_80_test_pred = np.clip(
    np.expm1(old_new_20_80_test_log),
    0,
    None
)

submission_old20_new80 = pd.DataFrame({
    "id": test["id"].values,
    "edi": old_new_20_80_test_pred
})

submission_old20_new80.to_csv(
    "submission_old20_new80.csv",
    index=False
)

print("Shape:", submission_old20_new80.shape)
print(
    "Missing:",
    submission_old20_new80["edi"].isna().sum()
)
print(
    "Infinite:",
    np.isinf(
        submission_old20_new80["edi"]
    ).sum()
)
print(
    "Minimum:",
    submission_old20_new80["edi"].min()
)
print(
    "Maximum:",
    submission_old20_new80["edi"].max()
)

display(submission_old20_new80.head())

assert submission_old20_new80.shape == (16000, 2)
assert submission_old20_new80["edi"].isna().sum() == 0
assert np.isfinite(
    submission_old20_new80["edi"]
).all()
assert submission_old20_new80["id"].equals(
    test["id"]
)

Shape: (16000, 2)
Missing: 0
Infinite: 0
Minimum: 4.024995217296752
Maximum: 724.5323579458573


,id,edi
0,24000,45.708526
1,24001,37.786515
2,24002,52.269762
3,24003,79.270877
4,24004,32.204950


In [47]:
comparison = pd.DataFrame({
    "old_model": np.expm1(final_blend_test_log),
    "new_model": np.expm1(final_ensemble_test_log),
    "old20_new80": submission_old20_new80["edi"].values
})

comparison["difference_from_old"] = (
    comparison["old20_new80"]
    - comparison["old_model"]
)

comparison["difference_from_new"] = (
    comparison["old20_new80"]
    - comparison["new_model"]
)

display(comparison.describe())

,old_model,new_model,old20_new80,difference_from_old,difference_from_new
count,16000.000000,16000.000000,16000.000000,16000.000000,16000.000000
mean,96.609657,96.501858,96.517731,-0.091925,0.015874
std,97.250727,96.793226,96.860154,3.456514,0.851389
min,3.965284,4.040035,4.024995,-68.459617,-11.144282
25%,29.742318,29.684920,29.733238,-0.603346,-0.159528
50%,63.397424,63.276384,63.292205,0.008668,-0.002168
75%,128.117306,127.533983,127.495873,0.632667,0.149910
max,737.329536,721.367872,724.532358,42.440264,15.783095


In [48]:
display(
    comparison[
        ["old_model", "new_model", "old20_new80"]
    ].corr()
)

,old_model,new_model,old20_new80
old_model,1.000000,0.999026,0.999374
new_model,0.999026,1.000000,0.999962
old20_new80,0.999374,0.999962,1.000000


In [49]:
expected_blend = np.clip(
    np.expm1(
        0.20 * final_blend_test_log
        + 0.80 * final_ensemble_test_log
    ),
    0,
    None
)

maximum_difference = np.max(
    np.abs(
        expected_blend
        - submission_old20_new80["edi"].values
    )
)

print("Maximum blend calculation difference:", maximum_difference)

assert maximum_difference < 1e-10

Maximum blend calculation difference: 0.0


In [50]:
comparison["absolute_change_from_old"] = (
    comparison["difference_from_old"].abs()
)

display(
    comparison.sort_values(
        "absolute_change_from_old",
        ascending=False
    ).head(20)
)

,old_model,new_model,old20_new80,difference_from_old,difference_from_new,absolute_change_from_old
13797,566.378210,482.135498,497.918593,-68.459617,15.783095,68.459617
3282,605.365733,539.558699,552.122365,-53.243369,12.563665,53.243369
10847,450.171326,388.531048,400.145534,-50.025792,11.614486,50.025792
583,602.755432,542.031732,553.667062,-49.088370,11.635330,49.088370
13982,555.495284,495.502223,506.959697,-48.535587,11.457474,48.535587
11898,343.736237,286.203434,296.885312,-46.850925,10.681878,46.850925
14847,679.084523,622.778401,633.653741,-45.430782,10.875340,45.430782
6914,465.253261,409.758747,420.302302,-44.950958,10.543556,44.950958
1123,515.363488,568.948034,557.803751,42.440264,-11.144282,42.440264
14582,577.826952,527.294789,537.035350,-40.791602,9.740561,40.791602


In [51]:
# ============================================================
# COMPLETE OOF COMPARISON
# ============================================================

old_oof_log = blend_oof_log
new_oof_log = best_new_oof_log

old20_new80_oof_log = (
    0.20 * old_oof_log
    + 0.80 * new_oof_log
)

old_oof_pred = np.clip(
    np.expm1(old_oof_log),
    0,
    None
)

new_oof_pred = np.clip(
    np.expm1(new_oof_log),
    0,
    None
)

old20_new80_oof_pred = np.clip(
    np.expm1(old20_new80_oof_log),
    0,
    None
)

oof_comparison_results = pd.DataFrame({
    "model": [
        "Old baseline",
        "New ensemble",
        "Old20_New80"
    ],

    "global_oof": [
        calculate_rmsle(y_raw, old_oof_pred),
        calculate_rmsle(y_raw, new_oof_pred),
        calculate_rmsle(y_raw, old20_new80_oof_pred)
    ],

    "test_like_oof": [
        calculate_rmsle(
            y_raw[very_test_like_mask],
            old_oof_pred[very_test_like_mask]
        ),
        calculate_rmsle(
            y_raw[very_test_like_mask],
            new_oof_pred[very_test_like_mask]
        ),
        calculate_rmsle(
            y_raw[very_test_like_mask],
            old20_new80_oof_pred[very_test_like_mask]
        )
    ],

    "young_hot_oof": [
        calculate_rmsle(
            y_raw[young_hot_mask],
            old_oof_pred[young_hot_mask]
        ),
        calculate_rmsle(
            y_raw[young_hot_mask],
            new_oof_pred[young_hot_mask]
        ),
        calculate_rmsle(
            y_raw[young_hot_mask],
            old20_new80_oof_pred[young_hot_mask]
        )
    ]
})

baseline_global = oof_comparison_results.loc[
    oof_comparison_results["model"] == "Old baseline",
    "global_oof"
].iloc[0]

baseline_test_like = oof_comparison_results.loc[
    oof_comparison_results["model"] == "Old baseline",
    "test_like_oof"
].iloc[0]

baseline_young_hot = oof_comparison_results.loc[
    oof_comparison_results["model"] == "Old baseline",
    "young_hot_oof"
].iloc[0]

oof_comparison_results["global_change"] = (
    oof_comparison_results["global_oof"]
    - baseline_global
)

oof_comparison_results["test_like_change"] = (
    oof_comparison_results["test_like_oof"]
    - baseline_test_like
)

oof_comparison_results["young_hot_change"] = (
    oof_comparison_results["young_hot_oof"]
    - baseline_young_hot
)

display(oof_comparison_results)

,model,global_oof,test_like_oof,young_hot_oof,global_change,test_like_change,young_hot_change
0,Old baseline,0.189636,0.230337,0.287495,0.000000,0.000000,0.000000
1,New ensemble,0.188875,0.228970,0.285908,-0.000761,-0.001367,-0.001587
2,Old20_New80,0.188834,0.228993,0.285971,-0.000802,-0.001344,-0.001524


In [54]:
fold_comparison_rows = []

for fold, (_, valid_idx) in enumerate(
    fixed_folds,
    start=1
):
    actual_fold = y_raw.iloc[valid_idx]

    old_fold_score = calculate_rmsle(
        actual_fold,
        old_oof_pred[valid_idx]
    )

    new_fold_score = calculate_rmsle(
        actual_fold,
        new_oof_pred[valid_idx]
    )

    blend_fold_score = calculate_rmsle(
        actual_fold,
        old20_new80_oof_pred[valid_idx]
    )

    fold_comparison_rows.append({
        "fold": fold,
        "old_baseline": old_fold_score,
        "new_ensemble": new_fold_score,
        "old20_new80": blend_fold_score,
        "new_change_vs_old": (
            new_fold_score - old_fold_score
        ),
        "blend_change_vs_old": (
            blend_fold_score - old_fold_score
        )
    })

fold_comparison_df = pd.DataFrame(
    fold_comparison_rows
)

display(fold_comparison_df)

,fold,old_baseline,new_ensemble,old20_new80,new_change_vs_old,blend_change_vs_old
0,1,0.193592,0.193166,0.193055,-0.000425,-0.000536
1,2,0.187897,0.187081,0.187035,-0.000816,-0.000862
2,3,0.187574,0.187043,0.186973,-0.000531,-0.000601
3,4,0.187554,0.186619,0.186596,-0.000935,-0.000958
4,5,0.191482,0.190381,0.190426,-0.001101,-0.001056


In [55]:
oof_prediction_comparison = pd.DataFrame({
    "actual_edi": y_raw.values,
    "old_prediction": old_oof_pred,
    "new_prediction": new_oof_pred,
    "old20_new80_prediction": old20_new80_oof_pred
})

oof_prediction_comparison["new_minus_old"] = (
    oof_prediction_comparison["new_prediction"]
    - oof_prediction_comparison["old_prediction"]
)

oof_prediction_comparison["blend_minus_old"] = (
    oof_prediction_comparison["old20_new80_prediction"]
    - oof_prediction_comparison["old_prediction"]
)

oof_prediction_comparison["old_abs_log_error"] = np.abs(
    np.log1p(oof_prediction_comparison["old_prediction"])
    - np.log1p(oof_prediction_comparison["actual_edi"])
)

oof_prediction_comparison["new_abs_log_error"] = np.abs(
    np.log1p(oof_prediction_comparison["new_prediction"])
    - np.log1p(oof_prediction_comparison["actual_edi"])
)

oof_prediction_comparison["blend_abs_log_error"] = np.abs(
    np.log1p(
        oof_prediction_comparison["old20_new80_prediction"]
    )
    - np.log1p(oof_prediction_comparison["actual_edi"])
)

display(
    oof_prediction_comparison.describe()
)

,actual_edi,old_prediction,new_prediction,old20_new80_prediction,new_minus_old,blend_minus_old,old_abs_log_error,new_abs_log_error,blend_abs_log_error
count,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000
mean,79.448278,77.794192,77.761386,77.764479,-0.032806,-0.029713,0.140686,0.140089,0.140056
std,82.364619,78.811455,78.576977,78.609605,2.963272,2.375556,0.127161,0.126687,0.126662
min,3.362100,3.317742,3.311396,3.312665,-85.081677,-69.049154,0.000006,0.000012,0.000007
25%,23.915800,24.606125,24.605918,24.653506,-0.599709,-0.480514,0.048702,0.048395,0.048167
50%,51.075700,50.770923,50.807294,50.783971,0.001618,0.001294,0.106926,0.105645,0.105503
75%,105.025925,103.133693,102.883088,102.911657,0.622947,0.497700,0.195309,0.195005,0.194923
max,796.341500,715.827506,715.259385,715.372973,53.396423,42.292928,1.228873,1.208400,1.212495


In [56]:
error_columns = [
    "old_abs_log_error",
    "new_abs_log_error",
    "blend_abs_log_error"
]

oof_prediction_comparison["best_model"] = (
    oof_prediction_comparison[error_columns]
    .idxmin(axis=1)
)

print(
    oof_prediction_comparison[
        "best_model"
    ].value_counts()
)

best_model
new_abs_log_error      11981
old_abs_log_error      11561
blend_abs_log_error      458
Name: count, dtype: int64


In [71]:
import os
import json
import numpy as np

BEST_DIR = (
    "/content/drive/MyDrive/"
    "EDI_Hackathon/best_020483"
)

os.makedirs(BEST_DIR, exist_ok=True)

# 1. Save submission
submission_old20_new80.to_csv(
    os.path.join(
        BEST_DIR,
        "submission_020483.csv"
    ),
    index=False
)

# 2. Save OOF arrays
np.save(
    os.path.join(
        BEST_DIR,
        "best_020483_oof_log.npy"
    ),
    old20_new80_oof_log
)

np.save(
    os.path.join(
        BEST_DIR,
        "old_baseline_oof_log.npy"
    ),
    blend_oof_log
)

np.save(
    os.path.join(
        BEST_DIR,
        "new_ensemble_oof_log.npy"
    ),
    best_new_oof_log
)

# 3. Save test arrays
np.save(
    os.path.join(
        BEST_DIR,
        "best_020483_test_log.npy"
    ),
    old_new_20_80_test_log
)

np.save(
    os.path.join(
        BEST_DIR,
        "old_baseline_test_log.npy"
    ),
    final_blend_test_log
)

np.save(
    os.path.join(
        BEST_DIR,
        "new_ensemble_test_log.npy"
    ),
    final_ensemble_test_log
)

# 4. Save metadata
best_model_metadata = {
    "kaggle_score": 0.20483,
    "global_oof": float(
        calculate_rmsle(
            y_raw,
            np.clip(
                np.expm1(old20_new80_oof_log),
                0,
                None
            )
        )
    ),
    "test_like_oof": float(
        calculate_rmsle(
            y_raw[very_test_like_mask],
            np.clip(
                np.expm1(old20_new80_oof_log),
                0,
                None
            )[very_test_like_mask]
        )
    ),
    "young_hot_oof": float(
        calculate_rmsle(
            y_raw[young_hot_mask],
            np.clip(
                np.expm1(old20_new80_oof_log),
                0,
                None
            )[young_hot_mask]
        )
    ),
    "feature_count": 58,
    "blend_space": "log",
    "final_weights": {
        "original_catboost": 0.16,
        "catboost_depth6_l2_10": 0.34,
        "catboost_depth7_l2_10": 0.34,
        "lightgbm": 0.16
    },
    "old_new_blend": {
        "old_weight": 0.20,
        "new_weight": 0.80
    },
    "model_iterations": {
        "original_catboost": 1840,
        "catboost_depth6_l2_10": 1932,
        "catboost_depth7_l2_10": 1786,
        "lightgbm": 1396
    },
    "seed": 42
}

with open(
    os.path.join(
        BEST_DIR,
        "best_020483_metadata.json"
    ),
    "w"
) as file:
    json.dump(
        best_model_metadata,
        file,
        indent=2
    )

print("Saved everything to:", BEST_DIR)

Saved everything to: /content/drive/MyDrive/EDI_Hackathon/best_020483


In [72]:
import os
import json
import numpy as np
import pandas as pd

BEST_DIR = (
    "/content/drive/MyDrive/"
    "EDI_Hackathon/best_020483"
)

required_files = [
    "submission_020483.csv",
    "best_020483_metadata.json",
    "best_020483_oof_log.npy",
    "best_020483_test_log.npy",
    "old_baseline_oof_log.npy",
    "new_ensemble_oof_log.npy",
    "old_baseline_test_log.npy",
    "new_ensemble_test_log.npy",
]

print("Checking folder:", BEST_DIR)
print()

missing_files = []

for filename in required_files:
    path = os.path.join(BEST_DIR, filename)

    if os.path.exists(path):
        print("FOUND:", filename)
    else:
        print("MISSING:", filename)
        missing_files.append(filename)

assert len(missing_files) == 0, (
    f"Missing files: {missing_files}"
)

print("\nAll required files exist.")

Checking folder: /content/drive/MyDrive/EDI_Hackathon/best_020483

FOUND: submission_020483.csv
FOUND: best_020483_metadata.json
FOUND: best_020483_oof_log.npy
FOUND: best_020483_test_log.npy
FOUND: old_baseline_oof_log.npy
FOUND: new_ensemble_oof_log.npy
FOUND: old_baseline_test_log.npy
FOUND: new_ensemble_test_log.npy

All required files exist.


In [73]:
metadata_path = os.path.join(
    BEST_DIR,
    "best_020483_metadata.json"
)

with open(metadata_path, "r") as file:
    saved_metadata = json.load(file)

print(json.dumps(saved_metadata, indent=2))

assert saved_metadata["kaggle_score"] == 0.20483
assert saved_metadata["feature_count"] == 58
assert saved_metadata["blend_space"] == "log"

print("\nMetadata is valid.")

{
  "kaggle_score": 0.20483,
  "global_oof": 0.18883394782919927,
  "test_like_oof": 0.2289931661342415,
  "young_hot_oof": 0.28597050162179655,
  "feature_count": 58,
  "blend_space": "log",
  "final_weights": {
    "original_catboost": 0.16,
    "catboost_depth6_l2_10": 0.34,
    "catboost_depth7_l2_10": 0.34,
    "lightgbm": 0.16
  },
  "old_new_blend": {
    "old_weight": 0.2,
    "new_weight": 0.8
  },
  "model_iterations": {
    "original_catboost": 1840,
    "catboost_depth6_l2_10": 1932,
    "catboost_depth7_l2_10": 1786,
    "lightgbm": 1396
  },
  "seed": 42
}

Metadata is valid.


In [74]:
saved_best_oof_log = np.load(
    os.path.join(
        BEST_DIR,
        "best_020483_oof_log.npy"
    )
)

saved_best_test_log = np.load(
    os.path.join(
        BEST_DIR,
        "best_020483_test_log.npy"
    )
)

saved_old_oof_log = np.load(
    os.path.join(
        BEST_DIR,
        "old_baseline_oof_log.npy"
    )
)

saved_new_oof_log = np.load(
    os.path.join(
        BEST_DIR,
        "new_ensemble_oof_log.npy"
    )
)

print("Best OOF shape:", saved_best_oof_log.shape)
print("Best test shape:", saved_best_test_log.shape)

assert saved_best_oof_log.shape == (24000,)
assert saved_best_test_log.shape == (16000,)

assert np.isfinite(saved_best_oof_log).all()
assert np.isfinite(saved_best_test_log).all()

print("\nSaved prediction arrays are valid.")

Best OOF shape: (24000,)
Best test shape: (16000,)

Saved prediction arrays are valid.


In [75]:
expected_best_oof_log = (
    0.20 * saved_old_oof_log
    + 0.80 * saved_new_oof_log
)

maximum_oof_difference = np.max(
    np.abs(
        saved_best_oof_log
        - expected_best_oof_log
    )
)

print(
    "Maximum saved OOF blend difference:",
    maximum_oof_difference
)

assert maximum_oof_difference < 1e-10

print("Saved OOF ensemble weights are correct.")

Maximum saved OOF blend difference: 0.0
Saved OOF ensemble weights are correct.


In [76]:
saved_best_oof_pred = np.clip(
    np.expm1(saved_best_oof_log),
    0,
    None
)

saved_global_oof = calculate_rmsle(
    y_raw,
    saved_best_oof_pred
)

saved_test_like_oof = calculate_rmsle(
    y_raw[very_test_like_mask],
    saved_best_oof_pred[very_test_like_mask]
)

saved_young_hot_oof = calculate_rmsle(
    y_raw[young_hot_mask],
    saved_best_oof_pred[young_hot_mask]
)

print("Saved global OOF:", saved_global_oof)
print("Saved test-like OOF:", saved_test_like_oof)
print("Saved young-hot OOF:", saved_young_hot_oof)

assert abs(saved_global_oof - 0.188834) < 0.00001
assert abs(saved_test_like_oof - 0.228993) < 0.00001
assert abs(saved_young_hot_oof - 0.285971) < 0.00001

print("\nSaved OOF checkpoints match the 0.20483 ensemble.")

Saved global OOF: 0.18883394782919927
Saved test-like OOF: 0.2289931661342415
Saved young-hot OOF: 0.28597050162179655

Saved OOF checkpoints match the 0.20483 ensemble.


In [77]:
saved_submission = pd.read_csv(
    os.path.join(
        BEST_DIR,
        "submission_020483.csv"
    )
)

print(saved_submission.shape)
display(saved_submission.head())

assert saved_submission.shape == (16000, 2)
assert saved_submission.columns.tolist() == ["id", "edi"]
assert saved_submission["id"].equals(test["id"])
assert saved_submission["id"].duplicated().sum() == 0
assert saved_submission["edi"].isna().sum() == 0
assert np.isfinite(saved_submission["edi"]).all()
assert (saved_submission["edi"] >= 0).all()

expected_test_predictions = np.clip(
    np.expm1(saved_best_test_log),
    0,
    None
)

maximum_submission_difference = np.max(
    np.abs(
        saved_submission["edi"].values
        - expected_test_predictions
    )
)

print(
    "Maximum submission difference:",
    maximum_submission_difference
)

assert maximum_submission_difference < 1e-8

print("\nEverything for the 0.20483 ensemble is saved correctly.")

(16000, 2)


,id,edi
0,24000,45.708526
1,24001,37.786515
2,24002,52.269762
3,24003,79.270877
4,24004,32.204950


Maximum submission difference: 5.684341886080802e-14

Everything for the 0.20483 ensemble is saved correctly.


In [78]:
import shutil

zip_path = shutil.make_archive(
    "/content/EDI_best_020483_backup",
    "zip",
    BEST_DIR
)

print("Backup created:", zip_path)

Backup created: /content/EDI_best_020483_backup.zip
